# BC–WA PM2.5 forecasting benchmark on Colab

Trains AirLapseV2 and baseline models on the BC + Washington hourly PM2.5
dataset that ships in this repo (`data/`), and scores 24-hour forecasts.

**Runtime → Change runtime type → GPU** before running. All metric files are
saved to Google Drive (`METRICS_DIR`), so a disconnected runtime loses no results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# metrics (benchmark.csv, per-run metric files, logs) -> Drive; the large
# prediction arrays and checkpoints stay on the runtime's local disk
METRICS_DIR = '/content/drive/MyDrive/bcwa_pm25_results'
LOCAL_RESULTS = '/content/results'

%cd /content
!rm -rf BC-WA-PM2.5-Forecast && git clone -q https://github.com/Grace-VN/BC-WA-PM2.5-Forecast.git
%cd /content/BC-WA-PM2.5-Forecast
!pip install -q -r requirements.txt
import os, torch
os.environ['METRICS_DIR'] = METRICS_DIR
os.environ['RESULTS_DIR'] = LOCAL_RESULTS
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

## Quick check (a few minutes)

One epoch of GRU, just to confirm everything runs. Its scores are meaningless.

In [ ]:
!METRICS_DIR="{METRICS_DIR}/smoke_test" python benchmark.py --models GRU --epochs 1 --repeats 1

## Benchmark

15 models, each trained **5 times** (seeds 0-4), reported as **mean ± std**.
The models are split into **4 groups of roughly equal run time** - run one
group per Colab session:

| Group | Models |
|---|---|
| 1 | MLP, GRU, LSTM, PM25_GNN, AirLapseV2, Crossformer, AirPhyNet, Transformer, AirDualODE |
| 2 | STMamba, Informer, AirFormer |
| 3 | Autoformer, TCN_DIR (Seo et al. 2026, re-implemented) |
| 4 | AirDDE |

Sessions can run **one after another or at the same time** (several Colab
tabs/runtimes), in any order, as long as they all write to the same Drive
folder (`METRICS_DIR`) and no two run the same group at once. Each finished
run is saved to Drive immediately (`scores/<model>__rep<k>.csv`); after a
disconnect, re-run the cell with the same group and it continues from the
run it stopped in. The free tier usually allows only one GPU runtime at a
time, so running groups in parallel may need Colab Pro or a second account
(in which case copy that account's `scores/` files into this Drive folder).


In [ ]:
GROUP = 1     # 1, 2, 3 or 4 - see the table above
!python benchmark.py --group {GROUP} --repeats 5 --epochs 50 --early_stop 10


In [ ]:
# Combine every group's results so far (run this any time, from any session)
!python benchmark.py --summary

# mean ± std over the runs per model (sample std), best RMSE first
import pandas as pd
s = pd.read_csv(f'{METRICS_DIR}/benchmark_summary.csv', index_col=0)
cols = ['RMSE', 'MAE', 'MAPE', 'RMSE_gt35.5', 'CSI@35.5', 'POD@35.5', 'FAR@35.5', 'CSI@55.5']
table = pd.DataFrame({'runs': s.n_repeats})
for c in cols:
    d = 1 if c == 'MAPE' else 3
    table[c] = [f'{m:.{d}f} ± {sd:.{d}f}' if pd.notna(sd) else f'{m:.{d}f}'
                for m, sd in zip(s[c + '_mean'], s[c + '_std'])]
table
